# 12 — Parcel descriptors and class-constrained alignment (Experiment 1 inputs)

Advisor's point (2026-08-28): a donor parcel may only stand in for a treated position of the
**same land type**. The raw 980-d arm compared identical coordinates across sites; here we
measure how wrong that was and build the permutation that fixes it.

* Parcel = cell (i, j) of the uniform 14×14 partition of the 101-px chip = latent parcel (i, j)
  (`panel_repr.py`, validity section; `panel_align.py`).
* Descriptor per parcel, pre-treatment only: NLCD 2021 class (mode; **hard constraint**),
  elevation, slope, aspect (sin, cos), TerraMind history mean and SD over clean P01–P08
  (parcel validity ≥ 0.5, ≥ 3 clean periods), 3×3-neighbour class histogram.
  *minimal* = class + elevation/slope + history mean (D = 7); *full* = all blocks (D = 29).
* Alignment: `scipy.optimize.linear_sum_assignment` on
  `C[p,q] = BIG·[class differs] + α·d²_phys + β·d²_terramind + γ·d²_context`, dummy columns so
  only unavoidable positions go unmatched (E1.10). Descriptors use P01–P08 only; the
  permutation is frozen before any P09/P10 chip is touched.


In [1]:
import sys, json
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
sys.path.insert(0, ".")
import panel_lib as pl, panel_repr as pr, panel_align as pa
pd.set_option("display.width", 220)

panel = pl.Panel.from_npz(pl.LATD / "latents_biweekly.npz")
VAL = pr.load_validity()
ALL_SITES = sorted(panel.roster["site_id"]); TREAT = panel.treatments; DON = pr.matched_donors(panel)

DESC = pa.build_descriptors(panel, VAL)               # E1.3–E1.7 defaults
pa.save_descriptors(DESC, "parcel_descriptors.npz")
S = pa.standardize(DESC)                                # E1.8 pooled over 60 sites x 196 parcels
print("descriptors:", len(DESC), "(site, sensor) entries; saved parcel_descriptors.npz")
nc = np.concatenate([DESC[(s, se)]["n_clean"] for s in ALL_SITES for se in pl.SENSORS])
print("clean P01-P08 periods per parcel (all sites, both sensors): "
      f"min {nc.min()} median {np.median(nc):.0f} max {nc.max()}; parcels < 3 clean: {(nc < 3).sum()}")


descriptors: 120 (site, sensor) entries; saved parcel_descriptors.npz
clean P01-P08 periods per parcel (all sites, both sensors): min 1 median 7 max 8; parcels < 3 clean: 283


## 1. Land-class composition of the 10 treated sites and their donors

Same-position agreement = share of the 196 positions where donor and treated carry the same NLCD class (what the raw-980 arm implicitly assumed = 1). Class-matchable = parcels that a hard class-constrained one-to-one assignment can pair.

In [2]:
rows = []
for t in TREAT:
    dt = DESC[(t, "sentinel1")]
    for j in DON[t]:
        dj = DESC[(j, "sentinel1")]
        ct, cj = pd.Series(dt["cls"]).value_counts(), pd.Series(dj["cls"]).value_counts()
        matchable = int(sum(min(ct.get(c, 0), cj.get(c, 0)) for c in ct.index if c != 0))
        rows.append({"treated": t, "donor": j, "same_position_agreement": pa.same_position_agreement(dt, dj),
                     "class_matchable": matchable,
                     "treated_top": ", ".join(f"{pa.NLCD_NAMES[c]} {n}" for c, n in ct.head(3).items()),
                     "donor_top": ", ".join(f"{pa.NLCD_NAMES[c]} {n}" for c, n in cj.head(3).items())})
AGREE = pd.DataFrame(rows); AGREE.to_csv("panel_align_agreement.csv", index=False)
print(AGREE.to_string(index=False))
print("\nsame-position agreement over the 50 (treated, donor) pairs: "
      f"min {AGREE.same_position_agreement.min():.2f} mean {AGREE.same_position_agreement.mean():.2f} "
      f"max {AGREE.same_position_agreement.max():.2f}")


       treated                  donor  same_position_agreement  class_matchable                            treated_top                               donor_top
treatment_0001 counterfactual_0001_01                 0.357143              141     pasture 83, deciduous 75, mixed 15    deciduous 105, pasture 47, dev-med 9
treatment_0001 counterfactual_0001_02                 0.408163              104     pasture 83, deciduous 75, mixed 15      pasture 173, deciduous 20, shrub 2
treatment_0001 counterfactual_0001_03                 0.397959              101     pasture 83, deciduous 75, mixed 15   pasture 177, deciduous 12, dev-open 5
treatment_0001 counterfactual_0001_04                 0.265306              142     pasture 83, deciduous 75, mixed 15     pasture 105, deciduous 49, grass 27
treatment_0001 counterfactual_0001_05                 0.418367              133     pasture 83, deciduous 75, mixed 15  pasture 139, deciduous 36, dev-open 13
treatment_0002 counterfactual_0002_01         

## 2. Alignment arms — matched parcels per (treated, donor) pair

Arms B (class only), C (+ elevation/slope), D (+ TerraMind history; minimal and full descriptor), E (full + context). Permutations saved to `parcel_perms.npz`, keyed `arm|sensor|treated|donor`.

In [3]:
ARMS = {"B": dict(),
        "C": dict(alpha=1.0),
        "D_min": dict(alpha=1.0, beta=1.0, variant="minimal"),
        "D_full": dict(alpha=1.0, beta=1.0, variant="full"),
        "E": dict(alpha=1.0, beta=1.0, gamma=1.0, variant="full")}
PERMS, mrows = {}, []
for arm, kw in ARMS.items():
    for sensor in pl.SENSORS:
        for t in TREAT:
            for j in DON[t]:
                p = pa.align_pair(S[(t, sensor)], S[(j, sensor)], **kw)
                PERMS[(arm, sensor, t, j)] = p
                mrows.append({"arm": arm, "sensor": sensor, "treated": t, "donor": j, "n_matched": int((p >= 0).sum())})
np.savez_compressed("parcel_perms.npz", **{"|".join(k): v for k, v in PERMS.items()})
MATCH = pd.DataFrame(mrows); MATCH.to_csv("panel_align_matched.csv", index=False)
print("matched parcels per pair (min / mean / max over the 50 pairs), by arm and sensor:")
print(MATCH.groupby(["arm", "sensor"]).n_matched.agg(["min", "mean", "max"]).round(1).to_string())
low = MATCH.query("arm == 'D_min' and n_matched < @pa.MIN_MATCHED")
print(f"\npairs below MIN_MATCHED = {pa.MIN_MATCHED} (arm D_min): {len(low)}")
if len(low): print(low.to_string(index=False))


/data/wang/junh/githubs/latent-synthetic-control/Satellite/notebooks/ts_SCM_ASCM/panel_align.py:188: RuntimeWarning: Mean of empty slice
  return np.nanmean(d * d, axis=-1)


matched parcels per pair (min / mean / max over the 50 pairs), by arm and sensor:
                  min   mean  max
arm    sensor                    
B      sentinel1   54  133.8  191
       sentinel2   54  127.8  191
C      sentinel1   54  133.8  191
       sentinel2   54  127.7  190
D_full sentinel1   54  133.8  191
       sentinel2   54  127.7  190
D_min  sentinel1   54  133.8  191
       sentinel2   54  127.7  190
E      sentinel1   54  133.8  191
       sentinel2   54  127.7  190

pairs below MIN_MATCHED = 60 (arm D_min): 7
  arm    sensor        treated                  donor  n_matched
D_min sentinel1 treatment_0004 counterfactual_0004_01         59
D_min sentinel1 treatment_0004 counterfactual_0004_03         54
D_min sentinel1 treatment_0005 counterfactual_0005_01         58
D_min sentinel2 treatment_0002 counterfactual_0002_04         57
D_min sentinel2 treatment_0004 counterfactual_0004_01         58
D_min sentinel2 treatment_0004 counterfactual_0004_03         54
D_min sent

## 3. Example — treated site 0001 (pasture / deciduous mosaic), donor 01, Sentinel-2

Left: NLCD class of each parcel. Right: the same donor's latent channel 0 at P05 before and after the arm-D (minimal) permutation. After alignment the donor's class map equals the treated map at every matched position by construction; unmatched positions are white.

In [4]:
t, j, sensor = "treatment_0001", DON["treatment_0001"][0], "sentinel2"
perm = PERMS[("D_min", sensor, t, j)]
cls_t = DESC[(t, sensor)]["cls"].reshape(14, 14); cls_j = DESC[(j, sensor)]["cls"].reshape(14, 14)
cls_al = np.where(perm >= 0, DESC[(j, sensor)]["cls"][np.maximum(perm, 0)], 0).reshape(14, 14)
classes = sorted(set(cls_t.ravel()) | set(cls_j.ravel())); cmap_idx = {c: k for k, c in enumerate(classes)}
to_idx = np.vectorize(lambda c: cmap_idx.get(c, -1))
A_raw = panel.L(j, sensor, 5).reshape(5, 196)[0].reshape(14, 14)
A_al = pa.aligned_vec(panel, j, sensor, 5, perm).reshape(5, 196)[0].reshape(14, 14)
A_t = panel.L(t, sensor, 5).reshape(5, 196)[0].reshape(14, 14)
fig, ax = plt.subplots(2, 3, figsize=(12, 8))
for a, m, ttl in zip(ax[0], [cls_t, cls_j, cls_al],
                     ["treated — NLCD class", "donor 01 — NLCD class (own layout)", "donor 01 — after alignment"]):
    mi = np.ma.masked_less(to_idx(m), 0)                 # unmatched / nodata -> white
    im = a.imshow(mi, cmap="tab20", vmin=-1, vmax=len(classes)); a.set_title(ttl); a.set_xticks([]); a.set_yticks([])
ax[0, 2].legend(handles=[plt.Rectangle((0, 0), 1, 1, color=plt.get_cmap("tab20")((cmap_idx[c] + 1) / (len(classes) + 1)))
                         for c in classes], labels=[pa.NLCD_NAMES[c] for c in classes], fontsize=7, loc="lower right")
vmin, vmax = np.nanmin([A_t, A_raw]), np.nanmax([A_t, A_raw])
for a, m, ttl in zip(ax[1], [A_t, A_raw, A_al],
                     ["treated — latent ch0, P05", "donor 01 — latent ch0, P05 (own layout)", "donor 01 — aligned"]):
    a.imshow(m, cmap="viridis", vmin=vmin, vmax=vmax); a.set_title(ttl); a.set_xticks([]); a.set_yticks([])
fig.suptitle(f"Sentinel-2, {t} vs {j}: same-position class agreement {pa.same_position_agreement(DESC[(t, sensor)], DESC[(j, sensor)]):.2f}, "
             f"matched after alignment {(perm >= 0).sum()}/196", fontsize=11)
fig.tight_layout(); fig.savefig("panel_align_example.png", dpi=130); plt.show()


## Reading

1. **The same-coordinate assumption was wrong at about half the positions.** Over the 50
   (treated, donor) pairs the same-position NLCD agreement is 0.07–0.94, mean ≈ 0.45. Mixed
   pasture/forest sites (0001, 0002, 0004, 0005) sit at 0.2–0.4; the forest-dominated 0007 at 0.7–0.9.
2. **Hard class matching pairs 54–191 of 196 parcels** (mean 134 Sentinel-1 / 128 Sentinel-2).
   Seven pairs fall below the 60-parcel floor (site 0004 donors 01/03, site 0005 donor 01 in both
   sensors, site 0002 donor 04 in Sentinel-2) and are dropped for that site — those donors are
   ~90 % deciduous where the treated site is half pasture. The cost terms (α, β, γ) do not change
   the matched count (it is fixed by the class histograms); they only choose *which* parcel of the
   class goes where.
3. Two donors carry NLCD nodata parcels inside the chip footprint (counterfactual_0004_05: 84,
   counterfactual_0003_03: 14); those parcels are never assigned.
4. 283 of 23,520 (site, sensor, parcel) descriptors have fewer than 3 clean P01–P08 periods
   (Sentinel-2 cloud gaps) and are treated as unmatched.
